# Traitement des fichiers bb_rr par blocs

In [1]:
import os
import pandas as pd
import numpy as np
from scipy import signal, interpolate

In [7]:
import os
import pandas as pd
import numpy as np

# Dossiers
dossier_rr = r"C:\Users\judupont\Desktop\bb_rr_tronque_blocs_v2"
dossier_filtre = r"C:\Users\judupont\Desktop\RtoR_v2_test"
os.makedirs(dossier_filtre, exist_ok=True)

# Paramètres
min_rr = 300
max_rr = 2000

fichiers = sorted([f for f in os.listdir(dossier_rr) if f.endswith(".csv")])

for fichier in fichiers:
    path_fichier = os.path.join(dossier_rr, fichier)
    df = pd.read_csv(path_fichier)
    
    rr_ms = []
    timestamps_rr = []
    prev_val = None
    prev_sign = None
    ts_first = None
    
    for _, row in df.iterrows():
        val = float(row["RtoR"])
        ts = row["Timestamp"]
        if val == 0:
            continue
        
        sign = 1 if val >= 0 else -1
        if prev_sign is not None and sign != prev_sign:
            rr = abs(prev_val) * 1000
            if min_rr <= rr <= max_rr:
                rr_ms.append(rr)
                timestamps_rr.append(ts_first)
            ts_first = ts
        elif prev_sign is None:
            ts_first = ts
        
        prev_val = val
        prev_sign = sign
    
    # dernier RR
    if prev_val is not None and ts_first is not None:
        rr = abs(prev_val) * 1000
        if min_rr <= rr <= max_rr:
            rr_ms.append(rr)
            timestamps_rr.append(ts_first)
    
    df_rr = pd.DataFrame({"Timestamp_RR": timestamps_rr, "rr_ms": rr_ms})
    if not df_rr.empty:
        df_rr.to_csv(os.path.join(dossier_filtre, fichier), index=False)

In [8]:
from scipy import signal, interpolate

# Paramètres des fenêtres
tv_window_sec = 300
tv_window_shift_sec = 60
interp_rate = 4
tv_effective_prc_threshold = 50

fichiers_rr = sorted([f for f in os.listdir(dossier_filtre) if f.endswith(".csv")])

for fichier in fichiers_rr:
    path_fichier = os.path.join(dossier_filtre, fichier)
    df_rr = pd.read_csv(path_fichier)
    
    rr_ms = df_rr["rr_ms"].values
    timestamps = df_rr["Timestamp_RR"].values
    time_sec = np.cumsum(rr_ms) / 1000
    start_time = time_sec[0]
    end_time = time_sec[-1]
    
    window_start = start_time
    results = []
    
    while window_start + tv_window_sec <= end_time:
        window_end = window_start + tv_window_sec
        idx = (time_sec >= window_start) & (time_sec < window_end)
        rr_window = rr_ms[idx]
        if len(rr_window) == 0:
            window_start += tv_window_shift_sec
            continue
        
        # Pourcentage RR valide
        pct_valid = 100 * np.sum((rr_window >= min_rr) & (rr_window <= max_rr)) / len(rr_window)
        if pct_valid < tv_effective_prc_threshold:
            window_start += tv_window_shift_sec
            continue
        
        # Interpolation et détending
        time_window = time_sec[idx]
        f_interp = interpolate.interp1d(time_window, rr_window, kind="cubic", fill_value="extrapolate")
        time_interp = np.arange(window_start, window_end, 1/interp_rate)
        rr_interp = f_interp(time_interp)
        rr_detrend = signal.detrend(rr_interp)
        
        # Métriques temps
        diff_rr = np.diff(rr_window)
        mean_rr = np.mean(rr_window)
        mean_hr = 60000 / mean_rr
        sdnn = np.std(rr_window, ddof=1)
        rmssd = np.sqrt(np.mean(diff_rr**2))
        nn50 = np.sum(np.abs(diff_rr) > 50)
        pnn50 = 100 * nn50 / len(diff_rr)
        
        # Métriques fréquence
        nperseg = min(256, len(rr_detrend))
        freqs, psd = signal.welch(rr_detrend, fs=interp_rate, window="hann", nperseg=nperseg, scaling="density")
        def band_power(low, high):
            idx_band = (freqs >= low) & (freqs < high)
            return np.trapz(psd[idx_band], freqs[idx_band])
        vlf = band_power(0.0, 0.04)
        lf  = band_power(0.04, 0.15)
        hf  = band_power(0.15, 0.4)
        
        results.append({
            "window_start_s": window_start,
            "window_end_s": window_end,
            "pct_valid_RR": pct_valid,
            "Mean_RR_ms": mean_rr,
            "Mean_HR_bpm": mean_hr,
            "SDNN_ms": sdnn,
            "RMSSD_ms": rmssd,
            "NN50": nn50,
            "pNN50_%": pnn50,
            "VLF_power_ms2": vlf,
            "LF_power_ms2": lf,
            "HF_power_ms2": hf,
            "LF_HF_ratio": lf/hf if hf > 0 else np.nan
        })
        
        window_start += tv_window_shift_sec
    
    df_metrics = pd.DataFrame(results)
    df_metrics.to_csv(os.path.join(dossier_filtre, fichier.replace(".csv","_metrics.csv")), index=False)

C:\Users\judupont\AppData\Local\Temp\ipykernel_21416\2009606536.py:59: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[idx_band], freqs[idx_band])
C:\Users\judupont\AppData\Local\Temp\ipykernel_21416\2009606536.py:59: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[idx_band], freqs[idx_band])
C:\Users\judupont\AppData\Local\Temp\ipykernel_21416\2009606536.py:59: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[idx_band], freqs[idx_band])
C:\Users\judupont\AppData\Local\Temp\ipykernel_21416\2009606536.py:59: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd[

KeyboardInterrupt: 

# STAT METRIQUE

In [29]:
"""
    hrv_frequency_domain calcule les métriques de HRV dans le domaine fréquentiel.

    - Un nombre minimal d'intervalles RR est requis (250) pour garantir
      une estimation spectrale fiable.
    - Les intervalles RR sont convertis en temps cumulatif afin d'obtenir
      l'instant de chaque battement cardiaque.
    - Le signal est ensuite interpolé à une fréquence régulière de 4 Hz
      afin de transformer la série RR irrégulière en série temporelle
      équidistante.
    - Le signal interpolé est "détrendé" pour supprimer les tendances lentes.
    - La densité spectrale de puissance est estimée à l'aide de la méthode
      de Welch.
    - La puissance spectrale est intégrée dans trois bandes de fréquence :
        * VLF : Very Low Frequency
        * LF : Low Frequency
        * HF : High Frequency
    - Les résultats sont renvoyés sous forme de dictionnaire.
"""

def hrv_time_domain(rr_ms):
    if len(rr_ms) < 250:
        return {}
    diff_rr = np.diff(rr_ms)
    return {
        "Mean_RR_ms": np.mean(rr_ms),
        "Mean_HR_bpm": 60000 / np.mean(rr_ms),
        "SDNN_ms": np.std(rr_ms, ddof=1),
        "RMSSD_ms": np.sqrt(np.mean(diff_rr ** 2)),
        "NN50": np.sum(np.abs(diff_rr) > 50),
        "pNN50_%": 100 * np.sum(np.abs(diff_rr) > 50) / len(diff_rr)
    }
def hrv_frequency_domain(rr_ms):
    if len(rr_ms) < 250:
        return {}

    time_ms = np.cumsum(rr_ms)
    time_ms = np.insert(time_ms, 0, 0)[:-1]

    fs = 4.0
    time_interp = np.arange(time_ms[0], time_ms[-1], 1000 / fs)

    f_interp = interpolate.interp1d(
        time_ms, rr_ms, kind="cubic", fill_value="extrapolate"
    )
    rr_interp = f_interp(time_interp)

    rr_detrend = signal.detrend(rr_interp)
    nperseg = min(256, len(rr_detrend))

    freqs, psd = signal.welch(
        rr_detrend,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        scaling="density"
    )

    def band_power(low, high):
        idx = (freqs >= low) & (freqs < high)
        return np.trapezoid(psd[idx], freqs[idx])

    vlf = band_power(0.0, 0.04)
    lf  = band_power(0.04, 0.15)
    hf  = band_power(0.15, 1)

    return {
        "VLF_power_ms2": vlf,
        "LF_power_ms2": lf,
        "HF_power_ms2": hf,
        "LF_HF_ratio": lf / hf if hf > 0 else np.nan
    }

In [13]:
def calculer_metriques_bloc(df_bloc):
    if "rr_ms_filt" not in df_bloc.columns:
        return None

    rr_ms = df_bloc["rr_ms_filt"].dropna().values

    if len(rr_ms) < 250:
        return None

    metriques = {}
    metriques.update(hrv_time_domain(rr_ms))
    metriques.update(hrv_frequency_domain(rr_ms))

    metriques["RR_conserves"] = len(rr_ms)
    metriques["RR_initiaux"] = df_bloc["rr_ms"].notna().sum()
    metriques["Pct_conserve"] = (
        100 * metriques["RR_conserves"] / max(metriques["RR_initiaux"], 1)
    )

    return metriques

In [14]:
dossier_rr = r"C:\Users\judupont\Desktop\RtoR_v2_test"
chemin_sortie = r"C:\Users\judupont\Desktop\HRV_RtoR_V2_python_test.csv"
chemin_df_global_txt = r"C:\Users\judupont\Desktop\df_global.txt"
 
# ======================================================================================
# CHARGEMENT df_global pour récuperer la condition ( Réduction de la menace ou Sandard)
# ======================================================================================
df_global = pd.read_csv(chemin_df_global_txt, sep="\t", encoding="utf-8")

# ===================
# LISTE DES FICHIERS
# ===================

fichiers = sorted([f for f in os.listdir(dossier_rr) if f.endswith(".csv")])
print(f"📂 {len(fichiers)} fichiers RR détectés\n")

resultats = []

# ========================
# BOUCLE SUR LES FICHIERS
# ========================
for i, fichier in enumerate(fichiers, 1):
    chemin = os.path.join(dossier_rr, fichier)
    print(f"[{i}/{len(fichiers)}] Calcul HRV : {fichier}")

    try:
        df = pd.read_csv(chemin)

        if "rr_ms_filt" not in df.columns:
            print("   ❌ Colonne rr_ms_filt absente → ignoré\n")
            continue

        metriques = calculer_metriques_bloc(df)
        if metriques is None:
            print("   ⚠️ Pas assez de RR filtrés → ignoré\n")
            continue

        nom = fichier.replace("_RtoR_v2.csv", "")
        pid, bloc = nom.split("_", 1)

        metriques["Numero_inclusion"] = pid
        metriques["Bloc"] = bloc.lower()

        resultats.append(metriques)
        print("   ✅ OK\n")

    except Exception as e:
        print(f"   ❌ ERREUR : {e}\n")

df_resultats = pd.DataFrame(resultats)

# =========================================
# AJOUT DE COLONNES FEUILLE ET CONDITION
# =========================================
df_resultats = df_resultats.merge(
    df_global[["Numero_inclusion", "Feuille", "Condition"]],
    on="Numero_inclusion",
    how="left"
)

# =================================
# TOUTE LES METRIQUES SUR 1 LIGNE  
# =================================
print("🔄 Conversion en format WIDE")

stats_bloc = [
    "Mean_RR_ms",
    "Mean_HR_bpm",
    "SDNN_ms",
    "RMSSD_ms",
    "NN50",
    "pNN50_%",
    "VLF_power_ms2",
    "LF_power_ms2",
    "HF_power_ms2",
    "LF_HF_ratio",
]

rows = []

for patient, df_p in df_resultats.groupby("Numero_inclusion"):
    ligne = {
        "Feuille": df_p["Feuille"].iloc[0],
        "Numero_inclusion": patient,
        "Condition": df_p["Condition"].iloc[0],
    }
    pct_blocs = []

    for bloc in ["bloc1", "bloc2", "bloc3"]:
        df_b = df_p[df_p["Bloc"] == bloc]

        # Nom du bloc
        ligne[f"{bloc.capitalize()}"] = bloc if not df_b.empty else ""

        for stat in stats_bloc:
            col = f"{stat}_{bloc.capitalize()}"
            if not df_b.empty and stat in df_b.columns:
                ligne[col] = df_b.iloc[0][stat]
            else:
                ligne[col] = np.nan

        # Stocker Pct_conserve pour calculer la moyenne finale
        if not df_b.empty and "Pct_conserve" in df_b.columns:
            pct_blocs.append(df_b.iloc[0]["Pct_conserve"])
        else:
            pct_blocs.append(np.nan)

    # =========================================
    # MOYENNE DE Pct_conserve SUR LES 3 BLOCS
    # =========================================
    ligne["Pct_conserve"] = np.nanmean(pct_blocs)

    rows.append(ligne)

df_final = pd.DataFrame(rows)

# ===============================
# ORDRE DES COLONNES DANS LE CSV
# ===============================

ordre_colonnes = [
    "Numero_inclusion",
    "Feuille",
    "Condition",

    "Mean_RR_ms_Bloc1",
    "Mean_HR_bpm_Bloc1",
    "SDNN_ms_Bloc1",
    "RMSSD_ms_Bloc1",
    "pNN50_%_Bloc1",
    "VLF_power_ms2_Bloc1",
    "LF_power_ms2_Bloc1",
    "HF_power_ms2_Bloc1",
    "LF_HF_ratio_Bloc1",

    "Mean_RR_ms_Bloc2",
    "Mean_HR_bpm_Bloc2",
    "SDNN_ms_Bloc2",
    "RMSSD_ms_Bloc2",
    "pNN50_%_Bloc2",
    "VLF_power_ms2_Bloc2",
    "LF_power_ms2_Bloc2",
    "HF_power_ms2_Bloc2",
    "LF_HF_ratio_Bloc2",

    "Mean_RR_ms_Bloc3",
    "Mean_HR_bpm_Bloc3",
    "SDNN_ms_Bloc3",
    "RMSSD_ms_Bloc3",
    "pNN50_%_Bloc3",
    "VLF_power_ms2_Bloc3",
    "LF_power_ms2_Bloc3",
    "HF_power_ms2_Bloc3",
    "LF_HF_ratio_Bloc3",

    "Pct_conserve",  
]

ordre_colonnes = [c for c in ordre_colonnes if c in df_final.columns]
df_final = df_final[ordre_colonnes]

df_final = df_final.sort_values(by="Pct_conserve", ascending=False).reset_index(drop=True)
df_final = df_final.round(2)
    
# ============
# SAUVEGARDE
# ============

df_final.to_csv(chemin_sortie, index=False, encoding="utf-8-sig")
print(f"\n💾 Résultats sauvegardés : {chemin_sortie}")
print("\n🏁 Calcul HRV terminé")

📂 378 fichiers RR détectés

[1/378] Calcul HRV : 0101CAR_bloc1_RtoR_v2.csv
   ✅ OK

[2/378] Calcul HRV : 0101CAR_bloc2_RtoR_v2.csv
   ✅ OK

[3/378] Calcul HRV : 0101CAR_bloc3_RtoR_v2.csv
   ✅ OK

[4/378] Calcul HRV : 0101EMS_bloc1_RtoR_v2.csv
   ✅ OK

[5/378] Calcul HRV : 0101EMS_bloc2_RtoR_v2.csv
   ✅ OK

[6/378] Calcul HRV : 0101EMS_bloc3_RtoR_v2.csv
   ✅ OK

[7/378] Calcul HRV : 0102PCR_bloc1_RtoR_v2.csv
   ✅ OK

[8/378] Calcul HRV : 0102PCR_bloc2_RtoR_v2.csv
   ✅ OK

[9/378] Calcul HRV : 0102PCR_bloc3_RtoR_v2.csv
   ✅ OK

[10/378] Calcul HRV : 0103BPS_bloc1_RtoR_v2.csv
   ✅ OK

[11/378] Calcul HRV : 0103BPS_bloc2_RtoR_v2.csv
   ✅ OK

[12/378] Calcul HRV : 0103BPS_bloc3_RtoR_v2.csv
   ✅ OK

[13/378] Calcul HRV : 0103SHS_bloc1_RtoR_v2.csv
   ✅ OK

[14/378] Calcul HRV : 0103SHS_bloc2_RtoR_v2.csv
   ✅ OK

[15/378] Calcul HRV : 0103SHS_bloc3_RtoR_v2.csv
   ✅ OK

[16/378] Calcul HRV : 0104IBS_bloc1_RtoR_v2.csv
   ✅ OK

[17/378] Calcul HRV : 0104IBS_bloc2_RtoR_v2.csv
   ✅ OK

[18/378] Cal